## Programmatic data uploads using the NOMAD API

In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd
import requests
import yaml

from pynxtools_apm import SEPARATOR
from pynxtools_apm.examples.deu_berlin_kuehbach.utils import get_project_id

print(os.getcwd())
with open(f"{os.getcwd()}{os.sep}/config.yaml", encoding="utf-8") as fp:
    config: dict[str, str] = yaml.safe_load(fp)
# for task, location in config.items():
#    print(f"{task}, {location}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{config['data']}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

with open(f"{config['data']}{os.sep}aaa_apm_nomad_project_names.yaml") as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

uploads: dict[str, str] = {}

## Upload to NOMAD via programmatically simulated drag-and-drop of the NeXus file one upload per entry

In [ ]:
# http://localhost:8000/nomad-oasis/gui/v2/apis
# https://nomad-lab.eu/prod/v1/staging/docs/howto/programmatic/publish_python.html

In [ ]:
def get_authentication_token(nomad_url: str, username: str, password: str) -> str:
    """Get the token for accessing your NOMAD unpublished uploads remotely."""
    try:
        # response = requests.get(
        #     f"{nomad_url}/auth/app_token", params={"username": username, "password": password, "expires_in": 86400}, timeout=10
        # )
        response = requests.post(
            f"{nomad_url}/auth/token",
            data={
                "grant_type": "password",
                "username": username,
                "password": password,
                "scope": "",
                "client_id": "string",
                "client_secret": "string",
            },
        )
        # print(response.json())

        token = response.json().get("access_token")
        if token:
            print(f"Generating authentication token OK")
            return token

        print(f"Generating authentication token failed")
        print(response.json())
        return ""
    except Exception:
        print(f"Generating authentication token failed")
        return ""

In [ ]:
def create_empty_upload(nomad_url: str, token: str, upload_name: str) -> str:
    """Create an empty upload to group a series of NOMAD entries (in NOMAD v2 aka a project)."""
    try:
        # response = requests.post(
        #     nomad_url + 'datasets/',
        #     headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'},
        #     json={"dataset_name": dataset_name},
        #     timeout=10
        #     )

        response = requests.post(
            f"{nomad_url}/uploads",
            headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
            params={
                "upload_name": upload_name,
                "embargo_length": 0,
                "auto_decompress": "true",
            },
            files={"file": None},
            timeout=10,
        )
        # print(response.json())

        upload_id = response.json().get("upload_id")
        if upload_id:
            print(f"Create empty upload {upload_name} OK")
            return upload_id

        print(f"Create empty upload {upload_name} failed")
        print(response.json())
        return ""
    except Exception:
        print(f"Create empty upload {upload_name} failed")
        return ""

In [ ]:
def reprocess_upload(nomad_url: str, upload_id: str) -> None:
    """Attempt to reprocess an upload."""
    try:
        response = requests.post(f"{nomad_url}/uploads/{upload_id}/action/process")
        print(response.json())
        print(f"Reprocessing upload {upload_id} OK")
        return
    except Exception:
        print(f"Reprocessing upload {upload_id} failed")
        return

In [ ]:
token = get_authentication_token(config["url"], config["username"], config["password"])
# print(token)

In [ ]:
project_range: tuple[int, int] = (1, 160)
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        if project_range[0] <= int(row.id) <= project_range[1]:
            project_id = get_project_id(row.id)

            # if int(row.id) != 6:
            #     continue

            if project_id in nomad_project_names:
                print(
                    f"project{SEPARATOR}{row.project_name}{SEPARATOR}{project_id}{SEPARATOR}upload..."
                )
                # plan what to copy
                queue: list[tuple[str, str]] = []

                # copy NeXus/HDF5 files from pynxtools into the upload
                file_paths: list[Path] = sorted(
                    [
                        str(path)
                        for path in Path(f"{config['pynxtools']}/run01").glob(
                            f"{project_id}*.nxs"
                        )
                    ]
                )
                print(len(file_paths))
                for file_path in file_paths:
                    queue.append((f"{file_path}", f"{file_path.rsplit('/')[-1]}"))
                del file_paths

                # copy NeXus/HDF5 files from paraprobe-toolbox into the upload
                for toolname in ("Ranger", "Surfacer"):
                    for artifact in ("Config", "Results"):
                        file_paths: list[Path] = sorted(
                            [
                                str(path)
                                for path in Path(f"{config['paraprobe']}").glob(
                                    f"{project_id}*{toolname}.{artifact}.SimID.1.nxs"
                                )
                            ]
                        )
                        print(f"{toolname}, {artifact}, {len(file_paths)}")
                        for file_path in file_paths:
                            queue.append(
                                (f"{file_path}", f"{file_path.rsplit('/')[-1]}")
                            )
                del toolname, artifact, file_paths

                if len(queue) > 0:
                    # create an empty upload
                    upload_name = f"{nomad_project_names[project_id]}"
                    print(upload_name)

                    upload_id = create_empty_upload(config["url"], token, upload_name)
                    print(upload_id)

                    # identify location in the filename for that upload to store all files of that upload
                    trg_directory = (
                        f"{config['nomad']}/{upload_id[0:2]}/{upload_id}/raw"
                    )
                    print(trg_directory)

                    for src, trg in queue:
                        # print(src)
                        # print(f"{trg_directory}/{trg}")
                        shutil.copy2(src, f"{trg_directory}/{trg}")
                    print(f"Instantiated {len(queue)}, {upload_name}, {upload_id}")

                    uploads[f"{project_id}"] = f"{upload_id}"
                    # reprocess_upload(config["url"], upload_id)
                    del upload_name, upload_id, trg_directory

                del queue

with open("upload_log.yml", "w") as fp:
    yaml.dump(
        uploads,
        fp,
        default_flow_style=False,  # block style
        allow_unicode=True,
        sort_keys=False,  # preserve key order
        indent=2,
    )

print(uploads)